In [1]:
import os
import json
import sys
import shutil

In [ ]:
dataset_dir = '/shared/workspace/lpv/mri_data/unprocessed/hca/'
t1w_dir = 'T1w_MPR_vNav_4e_RMS'
fmri_dirs = ['rfMRI_REST1_AP',  'rfMRI_REST1_PA', 'rfMRI_REST2_AP', 'rfMRI_REST2_PA']
for subject in os.listdir(dataset_dir):
    subject_dir = os.path.join(dataset_dir, subject)
    if not os.path.isdir(subject_dir):
        print("Skipping non-directory:", subject_dir)
        continue
    
    t1w_path = os.path.join(subject_dir, t1w_dir)
    if not os.path.exists(t1w_path):
        print("T1w directory not found for subject:", subject)
        continue
    fmri_paths = []
    for fmri_dir in fmri_dirs:
        fmri_path = os.path.join(subject_dir, fmri_dir)
        if os.path.exists(fmri_path):
            fmri_volume  = 
        else:
            print("fMRI directory not found for subject:", subject, "and fMRI dir:", fmri_dir)
    if not fmri_paths:
        print("No fMRI directories found for subject:", subject)
        continue
    print("Found fMRI paths for subject:", subject, fmri_paths)

Found fMRI paths for subject: HCA6429272_V1_MR ['/shared/workspace/lpv/mri_data/unprocessed/hca/HCA6429272_V1_MR/rfMRI_REST1_AP', '/shared/workspace/lpv/mri_data/unprocessed/hca/HCA6429272_V1_MR/rfMRI_REST1_PA', '/shared/workspace/lpv/mri_data/unprocessed/hca/HCA6429272_V1_MR/rfMRI_REST2_AP', '/shared/workspace/lpv/mri_data/unprocessed/hca/HCA6429272_V1_MR/rfMRI_REST2_PA']
Found fMRI paths for subject: HCA8921589_V1_MR ['/shared/workspace/lpv/mri_data/unprocessed/hca/HCA8921589_V1_MR/rfMRI_REST1_AP', '/shared/workspace/lpv/mri_data/unprocessed/hca/HCA8921589_V1_MR/rfMRI_REST1_PA', '/shared/workspace/lpv/mri_data/unprocessed/hca/HCA8921589_V1_MR/rfMRI_REST2_AP', '/shared/workspace/lpv/mri_data/unprocessed/hca/HCA8921589_V1_MR/rfMRI_REST2_PA']
Found fMRI paths for subject: HCA8405975_V1_MR ['/shared/workspace/lpv/mri_data/unprocessed/hca/HCA8405975_V1_MR/rfMRI_REST1_AP', '/shared/workspace/lpv/mri_data/unprocessed/hca/HCA8405975_V1_MR/rfMRI_REST1_PA', '/shared/workspace/lpv/mri_data/unpr

In [13]:
import os
import shutil
from pathlib import Path

dataset_dir = Path('/shared/workspace/lpv/mri_data/unprocessed/hcya/')
output_root = Path('/shared/workspace/lpv/synthetic_field_maps_ivan/data/unprocessed_synthbold_format/hcya/')
output_root.mkdir(parents=True, exist_ok=True)


# t1w_dir = "T1w_MPR_vNav_4e_RMS"
# fmri_dirs = ["rfMRI_REST1_AP", "rfMRI_REST1_PA", "rfMRI_REST2_AP", "rfMRI_REST2_PA"]

# hcya dataset
t1w_dir = "T1w_MPR1"
fmri_dirs = ["rfMRI_REST1_LR", "rfMRI_REST1_RL", "rfMRI_REST2_LR", "rfMRI_REST2_RL"]

def safe_symlink(src: Path, dst: Path):
    """
    Create or replace a symlink dst -> src.
    """
    dst.parent.mkdir(parents=True, exist_ok=True)

    if dst.is_symlink() or dst.exists():
        dst.unlink()

    # use absolute path so it works regardless of cwd
    dst.symlink_to(src.resolve())

for subject_dir in sorted(dataset_dir.iterdir()):
    if not subject_dir.is_dir():
        continue

    sid = subject_dir.name
    out_subj = output_root / sid
    out_subj.mkdir(exist_ok=True)

    # print(f"\n=== {sid} ===")

    # -------------------------
    # T1
    # -------------------------
    t1_folder = subject_dir / t1w_dir
    # print("Looking in:", t1_folder)
    # print("Files there:", list(t1_folder.iterdir()))
    if not t1_folder.exists():
        print(f"[SKIP] missing T1 folder: {t1_folder}")
        continue

    # Robust search: any T1w nii.gz in that folder
    t1_candidates = sorted(t1_folder.glob("*T1w*.nii.gz"))
    # if not t1_candidates:
    #     # fallback: any nii.gz
    #     t1_candidates = sorted(t1_folder.glob("*.nii.gz"))

    if not t1_candidates:
        print(f"[SKIP] no T1 nifti found in: {t1_folder}")
        continue

    t1_src = t1_candidates[0]
    t1_dst = out_subj / "T1w_acpc_dc.nii.gz"
    safe_symlink(t1_src, t1_dst)
    # print(f"[OK] T1 -> {t1_src.name}")

    # -------------------------
    # SBRefs
    # -------------------------
    found_any = False
    for fmri_dir in fmri_dirs:
        run_folder = subject_dir / fmri_dir
        if not run_folder.exists():
            print(f"[MISS] {fmri_dir} folder")
            continue

        sbref_candidates = sorted(run_folder.glob("*SBRef*.nii.gz"))
        if not sbref_candidates:
            print(f"[MISS] {fmri_dir} SBRef")
            continue

        sbref_src = sbref_candidates[0]
        sbref_dst = out_subj / f"{fmri_dir}_SBRef.nii.gz"
        safe_symlink(sbref_src, sbref_dst)
        # print(f"[OK] {fmri_dir}_SBRef -> {sbref_src.name}")
        found_any = True

    if not found_any:
        print("[WARN] no SBRefs found; leaving subject staged with only T1")

print("\nDone staging symlinks.")

[MISS] rfMRI_REST2_LR SBRef
[MISS] rfMRI_REST2_RL SBRef
[MISS] rfMRI_REST2_LR SBRef
[MISS] rfMRI_REST2_RL SBRef

Done staging symlinks.
